In [13]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [14]:
df = pd.read_csv("/content/Week3_GA_dataset.csv")

df.head()

,V1,V2,V3,V4,V5,Target
0,2.0,50.0,12500.0,98.0,NEGATIVE,YES
1,0.0,13.0,3250.0,28.0,NEGATIVE,YES
2,?,?,4000.0,35.0,NEGATIVE,YES
3,?,20.0,5000.0,45.0,NEGATIVE,YES
4,1.0,24.0,6000.0,77.0,NEGATIVE,NO


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   V1      748 non-null    object 
 1   V2      748 non-null    object 
 2   V3      748 non-null    float64
 3   V4      748 non-null    float64
 4   V5      748 non-null    object 
 5   Target  748 non-null    object 
dtypes: float64(2), object(4)
memory usage: 35.2+ KB


In [17]:

df.isnull().sum()

,0
V1,0
V2,0
V3,0
V4,0
V5,0
Target,0


In [18]:
df['V1'].unique()

array(['2.0', '0.0', '?', '1.0', '4.0', '5.0', '9.0', '3.0', '12.0',
       '6.0', '11.0', '10.0', '13.0', '8.0', '14.0', '7.0', '16.0',
       '15.0', '23.0', '21.0', '18.0', '22.0', '26.0', '35.0', '38.0',
       '40.0', '74.0', '20.0', '17.0', '25.0', '39.0', '72.0'],
      dtype=object)

In [19]:
df['V2'].unique()

array(['50.0', '13.0', '?', '20.0', '24.0', '12.0', '9.0', '46.0', '3.0',
       '10.0', '6.0', '5.0', '14.0', '11.0', '8.0', '16.0', '7.0', '2.0',
       '19.0', '4.0', '17.0', '1.0', '15.0', '22.0', '18.0', '38.0',
       '43.0', '34.0', '44.0', '26.0', '41.0', '21.0', '33.0'],
      dtype=object)

In [20]:
df.replace('?', np.nan, inplace= True)

In [21]:
df.isnull().sum()

,0
V1,5
V2,5
V3,0
V4,0
V5,0
Target,0


### Separate Features and Target

In [22]:
X = df.drop("Target", axis=1)
y = df['Target']

print(X.shape)

(748, 5)


### Convert V1 and V2 to numeric

In [23]:
X['V1'] = pd.to_numeric(X["V1"])
X['V2'] = pd.to_numeric(X["V2"])


### SimpleImputer

In [24]:
from sklearn.impute import SimpleImputer

imputer= SimpleImputer(strategy="mean")
X.iloc[:, [0,1]] = imputer.fit_transform(X.iloc[:, [0,1]])

print(X.shape)

(748, 5)


### StandardScaler

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X.iloc[:, [0,1,2,3]] = scaler.fit_transform(X.iloc[:, [0,1,2,3]])

print(X.shape)

(748, 5)


### OrdinalEncoder

In [26]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()

X.iloc[:, [4]] = encoder.fit_transform(X.iloc[:, [4]])

print(X.shape)

(748, 5)


In [29]:
print(X.var())

V1    1.001339
V2    1.001339
V3    1.001339
V4    1.001339
V5         0.0
dtype: object


### VarianceThreshold

In [27]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.1)

X_new = selector.fit_transform(X)

print(X_new.shape)

(748, 4)


### Removed features

In [32]:
print(X_new.shape)
print(selected_features)

(748, 4)
Index(['V1', 'V2', 'V3', 'V4'], dtype='object')


### Target Encoding

In [33]:
from sklearn.preprocessing import OrdinalEncoder

# Create an encoder for the target variable
target_encoder = OrdinalEncoder()

# encoding y, back to 1d
y = target_encoder.fit_transform(y.to_frame()).ravel()

# Check the first few encoded values
print(y[:10])


print(y.shape)

[1. 1. 1. 1. 0. 0. 1. 0. 1. 1.]
(748,)


In [34]:
print(target_encoder.categories_)

[array(['NO', 'YES'], dtype=object)]


### Train Logistic Regression

In [35]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X, y)

LogisticRegression()

### Recursive Feature Elimination (RFE)

In [36]:
from sklearn.feature_selection import RFE

# Create the RFE object
rfe = RFE(
    estimator=model,
    n_features_to_select=2
)

# Train RFE
rfe.fit(X, y)

RFE(estimator=LogisticRegression(), n_features_to_select=2)

In [37]:
print("Selected Features:")
print(X.columns[rfe.support_])

Selected Features:
Index(['V1', 'V3'], dtype='object')


### Sequential Forward Selection (SFS)

In [38]:
from sklearn.feature_selection import SequentialFeatureSelector

sfs = SequentialFeatureSelector(
    estimator=model,
    n_features_to_select=2,
    direction="forward"
)

sfs.fit(X, y)

SequentialFeatureSelector(estimator=LogisticRegression(),
                          n_features_to_select=2)

In [39]:
print(X.columns[sfs.get_support()])
print(sfs.get_support(indices=True))

Index(['V2', 'V4'], dtype='object')
[1 3]


### Sequential Feature Selection (Backward)

In [40]:
from sklearn.feature_selection import SequentialFeatureSelector

sfs_backward = SequentialFeatureSelector(
    estimator=model,
    n_features_to_select=2,
    direction="backward"
)

sfs_backward.fit(X, y)

SequentialFeatureSelector(direction='backward', estimator=LogisticRegression(),
                          n_features_to_select=2)

In [41]:
print("Selected Features:")
print(X.columns[sfs_backward.get_support()])

print("\nSelected Indices:")
print(sfs_backward.get_support(indices=True))

Selected Features:
Index(['V3', 'V4'], dtype='object')

Selected Indices:
[2 3]
